# Pivot Range Expansion Breakout on SPY
## Strategy Brief
The Pivot Range Expansion Breakout strategy aims to capture price movements when SPY breaks out of a defined range based on pivot points. The strategy predicts that significant price movements occur when the price breaks above or below the pivot range, indicating potential bullish or bearish trends. Trades are executed when the price breaks out of this range, entering long on an upward breakout and short on a downward breakout. Historical backtesting results suggest that this strategy can outperform a simple buy-and-hold approach under certain market conditions.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the trading context for our strategy, including the parameters and constants that will be used throughout the notebook.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-10'
TICKER = 'SPY'
PIVOT_LOOKBACK = 1  # Number of days to look back for pivot calculation
BREAKOUT_THRESHOLD = 0.01  # 1% breakout threshold

### PHASE 2 - Data Exploration
In this phase, we will download historical data for SPY from Yahoo Finance and compute the pivot range indicators. We will then plot these indicators overlaid on the price data to visualize potential breakout points.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Calculate pivot points
high = data['High'].shift(PIVOT_LOOKBACK)
low = data['Low'].shift(PIVOT_LOOKBACK)
close = data['Close'].shift(PIVOT_LOOKBACK)
pivot = (high + low + close) / 3

# Calculate breakout levels
resistance = pivot + (high - low)
support = pivot - (high - low)

# Plot price and pivot levels
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(pivot, label='Pivot', linestyle='--')
plt.plot(resistance, label='Resistance', linestyle='--')
plt.plot(support, label='Support', linestyle='--')
plt.title('SPY Price with Pivot Levels')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
In this phase, we define the signal generation logic based on the breakout of pivot levels. We will create a signal series to indicate potential entry and exit points.

In [ ]:
# Generate signals
signal = pd.Series(index=data.index, data=0)

# Long entry
signal[data['Close'] > resistance * (1 + BREAKOUT_THRESHOLD)] = 1

# Short entry
signal[data['Close'] < support * (1 - BREAKOUT_THRESHOLD)] = -1

# Positions
positions = signal.shift(1).fillna(0)

### PHASE 4 - Coding & Backtesting
In this phase, we will backtest the strategy by calculating daily returns based on the generated signals and plot the resulting equity curve.

In [ ]:
# Calculate daily returns
daily_returns = data['Close'].pct_change()
strategy_returns = daily_returns * positions

# Calculate equity curve
(1 + strategy_returns).cumprod().plot(figsize=(14, 7), label='Strategy')
(1 + daily_returns).cumprod().plot(label='Buy and Hold')
plt.title('Equity Curve')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
In this phase, we evaluate the performance of the strategy using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We will compare these metrics against a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (1 + returns).prod() ** (252 / len(returns)) - 1
    sharpe = returns.mean() / returns.std() * np.sqrt(252)
    downside_std = returns[returns < 0].std()
    sortino = returns.mean() / downside_std * np.sqrt(252)
    max_drawdown = (1 + returns).cumprod().div((1 + returns).cumprod().cummax()).min() - 1
    calmar = cagr / abs(max_drawdown)
    return cagr, sharpe, sortino, calmar, max_drawdown

strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd = calculate_performance_metrics(strategy_returns)
bh_cagr, bh_sharpe, bh_sortino, bh_calmar, bh_max_dd = calculate_performance_metrics(daily_returns)

# Comparison table
comparison = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe', 'Sortino', 'Calmar', 'Max Drawdown'],
    'Strategy': [strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd],
    'Buy and Hold': [bh_cagr, bh_sharpe, bh_sortino, bh_calmar, bh_max_dd]
})
print(comparison)

### PHASE 6 - Deploy & Monitor
In this phase, we create a function to download the last 60 days of SPY data, compute today's signal, and print the recommended position based on the strategy.

In [ ]:
def get_latest_signal():
    recent_data = yf.download(TICKER, period='60d')
    high = recent_data['High'].shift(PIVOT_LOOKBACK)
    low = recent_data['Low'].shift(PIVOT_LOOKBACK)
    close = recent_data['Close'].shift(PIVOT_LOOKBACK)
    pivot = (high + low + close) / 3
    resistance = pivot + (high - low)
    support = pivot - (high - low)
    latest_close = recent_data['Close'].iloc[-1]
    if latest_close > resistance.iloc[-1] * (1 + BREAKOUT_THRESHOLD):
        print('Long Position')
    elif latest_close < support.iloc[-1] * (1 - BREAKOUT_THRESHOLD):
        print('Short Position')
    else:
        print('No Position')

get_latest_signal()